# Onechess : essai de vitesse sur TPU (Colab)

Mesure la vitesse de l'entraîneur JAX d'Onechess sur le TPU gratuit de Colab : quelques minutes, rien n'est sauvé.

**À faire :** *Exécution → Modifier le type d'exécution → TPU v5e-1* (ou le TPU proposé), puis *Exécution → Tout exécuter*. Le résultat est la ligne `ESSAI DE VITESSE : … positions/s`. Aucune clé ni mot de passe n'est nécessaire.

In [ ]:
import subprocess, sys, os, glob, io, tarfile, urllib.request, time
try:
    import optax
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'optax'], check=True)
import jax
print(jax.__version__, jax.devices())
tarfile.open(fileobj=io.BytesIO(urllib.request.urlopen('https://codeload.github.com/ovnice-dev/onechess/tar.gz/main').read())).extractall('/content/code')
TOOLS = glob.glob('/content/code/*/tools')[0]
t0 = time.time()
subprocess.run([sys.executable, '-u', TOOLS + '/hf_sources.py', '--url',
                'https://huggingface.co/datasets/official-stockfish/master-binpacks/resolve/main/nodes5000pv2_UHO.binpack',
                '--prefixe', 'n5', '--dest', '/content/donnees', '--octets', '500000000', '--garde', '1.0', '--fixe', '240',
                '--graine', '11', '--morceau', '250000000', '--parts', str(os.cpu_count())], check=True)
print(f'donnees : {time.time()-t0:.0f} s, {len(glob.glob("/content/donnees/*.bin"))} fichiers')

In [ ]:
# Vitesse en 1024 neurones (le reseau final), lots de 16 384 puis de 65 536
for b in (16384, 65536):
    subprocess.run([sys.executable, '-u', TOOLS + '/train_jax.py', '--stream', '/content/donnees/*.bin', '--hidden', '1024',
                    '--buckets', 'mat4', '--out-buckets', '4', '--chunk-files', '4', '--batch', str(b),
                    '--max-pas', str(2000 if b == 16384 else 600), '--val', '100000'])

Pour comparaison : une T4 de Kaggle fait environ **157 000 positions/s** en 1024 (mesuré sur le réseau final).